# 2. Training & Evaluation

This notebook covers the three ways to turn preprocessed data into scored
models: training, the two evaluation protocols (internal cross-validation
and external generalization), and the `benchmark()` convenience method that
chains all of it into one call.


> **Prerequisites**
> - STP-Bench installed (`bash scripts/create_env.sh`, see the
>   [README](../README.md#installation)).
> - Benchmark data downloaded for the dataset(s) used below (see
>   [README — Benchmark Data](../README.md#benchmark-data)), or your own
>   dataset added following
>   [docs/guide.md — Adding a New Dataset](../docs/guide.md#adding-a-new-dataset).
> - Run this notebook from the repo root, or pass `repo_root=` explicitly to
>   `STPred(...)`.


In [ ]:
from stpbench import STPred

stp = STPred(models=["StNet"], repo_root=".")
stp.preprocess(data="ncche/xenium")   # skipped if already done — see notebook 1


## Training

`stp.train(data=...)` trains every model configured on `stp` across the
dataset's cross-validation folds (`TRAINING.num_k` in the data config), using
each model's own `config/model/<Model>.yaml` hyperparameters.


In [ ]:
result = stp.train(data="ncche/xenium")


`data` defaults to whatever `preprocess()`/`train()` most recently ran
against on this instance, so a second call can omit it:

```python
stp.train()  # reuses "ncche/xenium" from the call above
```

### Reading the result

Every workflow method (except `check()`/`visualize()`) returns a
`BenchmarkResult` — dict-compatible, with a few convenience views:


In [ ]:
result.summary()          # per-model results, cross-fold aggregate stats (mean, std, per_fold)
result.to_records()        # flat list of per-fold dicts
result.to_dataframe()       # the same, as a pandas DataFrame
result.best_checkpoints()   # best checkpoint path per model/fold


In [ ]:
result.save("training_results.csv")


### Where things land

- Logs: `<GENERAL.log_path>/<data>/<model>/<timestamp>/`
- Checkpoints: `<GENERAL.log_path>/<data>/<model>/<timestamp>/fold<k>/`

`stp` remembers the checkpoint timestamp per model/fold from this call, which
is what lets `evaluate_internal()` / `predict()` below omit `timestamps=` and
just resolve "the run I just trained."

### Training multiple models side by side

Every model configured on `stp` trains under the same `preprocess()` plan
and the same CV folds — the point of the benchmark being that results stay
directly comparable.


In [ ]:
stp_multi = STPred(models=["StNet", "TRIPLEX"], repo_root=".")
stp_multi.preprocess(data="ncche/xenium")
multi_result = stp_multi.train(data="ncche/xenium")
multi_result.to_dataframe()


### Tracking runs with Weights & Biases

Opt in at construction time; every subsequent `train()` call on that
instance logs to the given project.

```python
stp_tracked = STPred(models=["StNet"], repo_root=".", wandb=True, wandb_project="ST_prediction")
stp_tracked.train(data="ncche/xenium")
```


## Internal Evaluation ("internal test")

`stp.evaluate_internal(data=...)` scores each trained checkpoint on its own
held-out cross-validation test fold — the "internal test" protocol every
model in STP-Bench is compared under. `data` defaults to the last-trained
dataset, so right after `train()` it can be omitted.


In [ ]:
eval_res = stp.evaluate_internal()
eval_res.summary()          # per-model metrics, aggregated across folds
eval_res.to_dataframe()
eval_res.prediction_dirs()   # where each fold's predicted .h5ad files were written


In [ ]:
eval_res.save("internal_eval_results.csv")


### Pinning a specific training run

By default, `evaluate_internal()` resolves the *latest* run directory per
model under `<log_path>/<data>/<model>/`. Pass `timestamps=` to pin an
older/specific run instead — a single string (applied to every model), or a
`{model: timestamp}` dict. `folds=` restricts to a subset of folds — useful
when some are still training and you want a partial read on progress.


In [ ]:
stp.evaluate_internal(
    data="ncche/xenium",
    timestamps={"StNet": "2026-05-18-12-00-00"},
    folds=[0],
)


## External Evaluation ("external test")

`stp.evaluate_external(data, train_data=...)` scores checkpoints trained on
one dataset (`train_data`) against a *different*, labeled dataset (`data`) —
the generalization test companion to internal cross-validation above.

"Labeled" here means the external dataset has real ground-truth ST
expression to compare against — unlike the WSI-only inference covered in
notebook 3.


In [ ]:
ext_res = stp.evaluate_external(data="hest/LUAD", train_data="ncche/xenium")


`train_data` defaults to the instance's internal dataset (the last one
`preprocess()`/`train()` ran against), so right after training it's often
enough to write `stp.evaluate_external(data="hest/LUAD")`.

### What happens automatically

- If `hest/LUAD` is missing its base artifacts (patches/embeddings),
  `preprocess()` runs for it first — no separate step needed.
- Any model-specific `extra_preprocess` (graph construction, reference
  banks, similarity matrices — e.g. EGN/EGGN/OmiCLIP) re-runs in **external**
  mode, correctly namespaced against the *training* run rather than being
  rebuilt as if `hest/LUAD` were its own standalone training dataset.


In [ ]:
ext_res.summary()
ext_res.to_dataframe()
ext_res.save("external_eval_results.csv")


Same `timestamps=`/`folds=` pinning mechanism as internal evaluation
applies here too. Both `evaluate_internal()` and `evaluate_external()` are
thin wrappers over one shared method, if you'd rather branch on `mode`
yourself:

```python
stp.evaluate(mode="int", data="ncche/xenium")
stp.evaluate(mode="ext", data="hest/LUAD", external_data="hest/LUAD")
```


## One-shot Benchmark

`stp.benchmark(internal_data, external_data=None)` chains everything above
into a single call: `preprocess` &rarr; `train` &rarr; `evaluate_internal`,
and — if `external_data` is given — also `preprocess` &rarr; `predict` &rarr;
`evaluate_external` against it. This is the shortest path to a full
internal + external benchmark run for a new model or dataset.


In [ ]:
stp_bm = STPred(models=["StNet"], repo_root=".", gpu=1)

bm_result = stp_bm.benchmark(
    internal_data="ncche/xenium",
    external_data="hest/LUAD",
)


### Reading the combined result

`benchmark()`'s result additionally exposes each individual step's own
`BenchmarkResult` under `result["steps"]`:


In [ ]:
bm_result.summary()   # combined internal + external summary

bm_result["steps"]["train"].summary()
bm_result["steps"]["evaluate_internal"].summary()
bm_result["steps"]["evaluate_external"].summary()


In [ ]:
bm_result.save("benchmark_results.csv")


### Internal-only, overrides, and dry-run

Omit `external_data` to run just `preprocess` &rarr; `train` &rarr;
`evaluate_internal`. Extra keyword arguments are forwarded to the
`preprocess()` calls `benchmark()` makes internally. `dry_run=True` returns
the computed plan without executing anything — the same mechanism as
`preprocess()`'s own dry-run from notebook 1.


In [ ]:
internal_only = stp_bm.benchmark(internal_data="ncche/xenium")

stp_bm.benchmark(internal_data="ncche/xenium", external_data="hest/LUAD", overwrite=True)

plan = stp_bm.benchmark(internal_data="ncche/xenium", external_data="hest/LUAD", dry_run=True)
plan
